# FLUKE Sentiment Analysis with Qwen3 Model Scaling Experiment

This notebook evaluates sentiment analysis robustness across different Qwen3 model sizes (0.6B to 235B-A22B) using FLUKE linguistic modifications.

In [ ]:
from datasets import load_dataset
import dspy
import openai
import os
import re
import pandas as pd
import json
import random
from dotenv import load_dotenv
import glob
from scipy import stats
import time
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
load_dotenv()

In [ ]:
openai.api_key = os.getenv('OPENAI_API_KEY')
openai.organization = os.getenv('OPENAI_ORGANIZATION')

# Qwen3 model family with varying sizes (using OpenRouter)
QWEN3_MODELS = {
    'qwen3-0.6b': {
        'id': 'openrouter/qwen/qwen3-0.6b-04-28',
        'size': '0.6B',
        'params': 600_000_000,
        'description': 'Smallest Qwen3 variant'
    },
    'qwen3-1.7b': {
        'id': 'openrouter/qwen/qwen3-1.7b',
        'size': '1.7B',
        'params': 1_700_000_000,
        'description': 'Small Qwen3 variant'
    },
    'qwen3-4b': {
        'id': 'openrouter/qwen/qwen3-4b:free',
        'size': '4B',
        'params': 4_000_000_000,
        'description': 'Small-medium Qwen3 variant (free)'
    },
    'qwen3-8b': {
        'id': 'openrouter/qwen/qwen3-8b-04-28',
        'size': '8B',
        'params': 8_000_000_000,
        'description': 'Medium Qwen3 variant'
    },
    'qwen3-14b': {
        'id': 'openrouter/qwen/qwen3-14b-04-28',
        'size': '14B',
        'params': 14_000_000_000,
        'description': 'Medium-large Qwen3 variant'
    },
    'qwen3-32b': {
        'id': 'openrouter/qwen/qwen3-32b-04-28',
        'size': '32B',
        'params': 32_000_000_000,
        'description': 'Large Qwen3 variant'
    },
    'qwen3-30b-a3b': {
        'id': 'openrouter/qwen/qwen3-30b-a3b-04-28',
        'size': '30B-A3B',
        'params': 30_000_000_000,
        'description': 'Large Qwen3 mixture variant'
    },
    'qwen3-235b-a22b': {
        'id': 'openrouter/qwen/qwen3-235b-a22b-04-28',
        'size': '235B-A22B',
        'params': 235_000_000_000,
        'description': 'Largest Qwen3 variant (mixture of experts)'
    }
}

# Display model configuration
print("Qwen3 Model Scaling Configuration (OpenRouter):")
print("=" * 50)
for name, config in QWEN3_MODELS.items():
    print(f"{name:15} | {config['size']:10} | {config['description']}")

print(f"\nTotal models to test: {len(QWEN3_MODELS)}")
print(f"Parameter range: {QWEN3_MODELS['qwen3-0.6b']['params']:,} to {QWEN3_MODELS['qwen3-235b-a22b']['params']:,}")

In [ ]:
# Qwen3 model family with varying sizes
QWEN3_MODELS = {
    'qwen3-0.6b': {
        'id': 'together_ai/Qwen/Qwen2.5-0.5B-Instruct',  # Using closest available
        'size': '0.6B',
        'params': 600_000_000,
        'description': 'Smallest Qwen3 variant'
    },
    'qwen3-1.7b': {
        'id': 'together_ai/Qwen/Qwen2.5-1.5B-Instruct',  # Using closest available
        'size': '1.7B',
        'params': 1_700_000_000,
        'description': 'Small Qwen3 variant'
    },
    'qwen3-4b': {
        'id': 'together_ai/Qwen/Qwen2.5-3B-Instruct',  # Using closest available
        'size': '4B',
        'params': 4_000_000_000,
        'description': 'Small-medium Qwen3 variant'
    },
    'qwen3-8b': {
        'id': 'together_ai/Qwen/Qwen2.5-7B-Instruct',  # Using closest available
        'size': '8B',
        'params': 8_000_000_000,
        'description': 'Medium Qwen3 variant'
    },
    'qwen3-14b': {
        'id': 'together_ai/Qwen/Qwen2.5-14B-Instruct',
        'size': '14B',
        'params': 14_000_000_000,
        'description': 'Medium-large Qwen3 variant'
    },
    'qwen3-32b': {
        'id': 'together_ai/Qwen/Qwen2.5-32B-Instruct',
        'size': '32B',
        'params': 32_000_000_000,
        'description': 'Large Qwen3 variant'
    },
    'qwen3-70b': {
        'id': 'together_ai/Qwen/Qwen2.5-72B-Instruct',  # Using closest available
        'size': '70B',
        'params': 70_000_000_000,
        'description': 'Very large Qwen3 variant'
    },
    'qwen3-235b': {
        'id': 'together_ai/Qwen/QwQ-32B-Preview',  # Using available large model
        'size': '235B-A22B',
        'params': 235_000_000_000,
        'description': 'Largest Qwen3 variant (mixture of experts)'
    }
}

# Display model configuration
print("Qwen3 Model Scaling Configuration:")
print("=" * 50)
for name, config in QWEN3_MODELS.items():
    print(f"{name:15} | {config['size']:10} | {config['description']}")

print(f"\nTotal models to test: {len(QWEN3_MODELS)}")

In [ ]:
# Load dataset
ds = load_dataset('stanfordnlp/sst2')['validation']
print(f"Dataset size: {len(ds)}")

# Use smaller subset for scaling experiment
SAMPLE_SIZE = 50  # Adjust based on computational budget
print(f"Using sample size: {SAMPLE_SIZE}")

In [ ]:
def remove_space(text):
    """Clean up spacing and formatting in text."""
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        # Remove multiple spaces
        cleaned = ' '.join(line.split())
        
        # Fix spacing around punctuation
        cleaned = re.sub(r'\s+([.,!?:;])', r'\1', cleaned)
        cleaned = re.sub(r'([.,!?:;])\s+', r'\1 ', cleaned)
        
        # Fix contractions
        cleaned = re.sub(r'\s*\'\s*s\b', "'s", cleaned)
        cleaned = re.sub(r'\s*n\s*\'\s*t\b', "n't", cleaned)
        cleaned = re.sub(r'\s*\'\s*ve\b', "'ve", cleaned)
        cleaned = re.sub(r'\s*\'\s*re\b', "'re", cleaned)
        cleaned = re.sub(r'\s*\'\s*ll\b', "'ll", cleaned)
        cleaned = re.sub(r'\s*\'\s*d\b', "'d", cleaned)
        cleaned = re.sub(r'\s*\'\s*m\b', "'m", cleaned)
        
        # Fix spaces around parentheses
        cleaned = re.sub(r'\(\s+', '(', cleaned)
        cleaned = re.sub(r'\s+\)', ')', cleaned)
        
        # Remove leading/trailing whitespace
        cleaned = cleaned.strip()
        cleaned_lines.append(cleaned)
        
    return '\n'.join(cleaned_lines)

In [ ]:
examples = [
    dspy.Example({ 
                  "text": remove_space(r["sentence"]), 
                  "label": r["label"]}
                  ).with_inputs("text") 
    for r in ds
]

In [ ]:
example = examples[835]
for k, v in example.items():
    print(f"\n{k.upper()}:\n")
    print(v)

In [ ]:
def extract_prediction(text):
    """Extract prediction from model output."""
    # Look for explicit answers first
    patterns = [
        r'Answer:\s*([01])',
        r'Final answer:\s*([01])',
        r'Label:\s*([01])',
        r'Prediction:\s*([01])',
        r'Classification:\s*([01])',
        r'\b([01])\b'
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1]
    
    return ""

In [ ]:
def eval_metric(true, prediction, trace=None):
    """Evaluate sentiment prediction."""
    pred = prediction.label
    parsed_answer = extract_prediction(pred)
    return parsed_answer == str(true.label)

# Evaluate across all Qwen3 model sizes

In [ ]:
from dspy.evaluate import Evaluate

## Sentiment Classification Across Model Sizes

In [ ]:
class QwenSentiment(dspy.Signature):
    """Classify sentiment of the given text. Answer with 1 for positive sentiment, 0 for negative sentiment."""
    text = dspy.InputField()
    label = dspy.OutputField(prefix='Answer:')

class QwenSentimentModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(QwenSentiment)

    def forward(self, text):
        return self.prog(text=text)

## Evaluate Original Test Set Across All Model Sizes

In [ ]:
# Use smaller subset for scaling experiment
test_examples = examples[:SAMPLE_SIZE]

print(f"Evaluating on {len(test_examples)} examples across {len(QWEN3_MODELS)} models")

# Store results for all models
model_results = {}
scaling_analysis = []

for model_name, model_config in QWEN3_MODELS.items():
    print(f"\n{'='*60}")
    print(f"Evaluating {model_name} ({model_config['size']})")
    print(f"Model ID: {model_config['id']}")
    print(f"{'='*60}")
    
    try:
        # Configure model
        lm = dspy.LM(model_config['id'], temperature=0, max_tokens=100)
        dspy.configure(lm=lm)
        
        # Initialize module
        qwen_sentiment = QwenSentimentModule()
        
        # Evaluate model
        evaluate = Evaluate(
            devset=test_examples, 
            metric=eval_metric, 
            num_threads=1,  # Conservative for stability
            display_progress=True, 
            display_table=5, 
            return_outputs=True, 
            return_all_scores=True
        )
        
        start_time = time.time()
        results = evaluate(qwen_sentiment)
        evaluation_time = time.time() - start_time
        
        accuracy = results[0]
        
        # Store detailed results
        items = []
        for sample in results[1]:
            item = {
                'text': sample[0]['text'],
                'label': sample[0]['label'],
                'pred': extract_prediction(sample[1]['label']),
                'raw_output': sample[1]['label']
            }
            items.append(item)
        
        df_result = pd.DataFrame(data=items)
        output_file = f'results/sa/{model_name}-0shot-sst2.csv'
        df_result.to_csv(output_file, index=False)
        print(f"Results saved to: {output_file}")
        
        # Store results for scaling analysis
        model_results[model_name] = {
            'config': model_config,
            'accuracy': accuracy,
            'evaluation_time': evaluation_time,
            'samples': len(test_examples),
            'results_df': df_result
        }
        
        scaling_analysis.append({
            'model_name': model_name,
            'model_size': model_config['size'],
            'params': model_config['params'],
            'accuracy': accuracy,
            'evaluation_time': evaluation_time,
            'samples': len(test_examples),
            'throughput': len(test_examples) / evaluation_time if evaluation_time > 0 else 0
        })
        
        print(f"Accuracy: {accuracy:.3f}")
        print(f"Evaluation time: {evaluation_time:.1f}s")
        
        # Add delay between models to respect rate limits
        time.sleep(5)
        
    except Exception as e:
        print(f"Error evaluating {model_name}: {e}")
        continue

print(f"\n{'='*60}")
print("Scaling experiment completed!")
print(f"Successfully evaluated {len(model_results)} models")

## Scaling Analysis and Visualization

In [ ]:
# Create scaling analysis dataframe
if scaling_analysis:
    scaling_df = pd.DataFrame(scaling_analysis)
    scaling_df = scaling_df.sort_values('params')  # Sort by model size
    
    print("Qwen3 Model Scaling Results:")
    print("=" * 60)
    display_cols = ['model_name', 'model_size', 'accuracy', 'evaluation_time', 'throughput']
    print(scaling_df[display_cols].round(3))
    
    # Save scaling results
    scaling_df.to_csv('results/sa/qwen3-scaling-analysis.csv', index=False)
    print(f"\nScaling analysis saved to: results/sa/qwen3-scaling-analysis.csv")
else:
    print("No scaling analysis data available")

In [ ]:
# Visualize scaling behavior
if scaling_analysis and len(scaling_analysis) > 1:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Plot 1: Accuracy vs Model Size
    axes[0, 0].plot(scaling_df['params'], scaling_df['accuracy'], 'bo-', linewidth=2, markersize=8)
    axes[0, 0].set_xscale('log')
    axes[0, 0].set_xlabel('Parameters (log scale)')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].set_title('Accuracy vs Model Size')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Annotate points
    for idx, row in scaling_df.iterrows():
        axes[0, 0].annotate(row['model_size'], 
                           (row['params'], row['accuracy']), 
                           xytext=(5, 5), textcoords='offset points', 
                           fontsize=9, alpha=0.8)
    
    # Plot 2: Throughput vs Model Size
    axes[0, 1].plot(scaling_df['params'], scaling_df['throughput'], 'ro-', linewidth=2, markersize=8)
    axes[0, 1].set_xscale('log')
    axes[0, 1].set_xlabel('Parameters (log scale)')
    axes[0, 1].set_ylabel('Throughput (samples/sec)')
    axes[0, 1].set_title('Throughput vs Model Size')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Accuracy vs Throughput Trade-off
    axes[1, 0].scatter(scaling_df['throughput'], scaling_df['accuracy'], 
                      s=100, c=np.log10(scaling_df['params']), cmap='viridis', alpha=0.7)
    axes[1, 0].set_xlabel('Throughput (samples/sec)')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].set_title('Accuracy vs Throughput Trade-off')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Add colorbar
    cbar = plt.colorbar(axes[1, 0].collections[0], ax=axes[1, 0])
    cbar.set_label('log10(Parameters)')
    
    # Plot 4: Model comparison bar chart
    axes[1, 1].bar(range(len(scaling_df)), scaling_df['accuracy'], 
                   color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f'][:len(scaling_df)])
    axes[1, 1].set_xlabel('Model')
    axes[1, 1].set_ylabel('Accuracy')
    axes[1, 1].set_title('Model Comparison')
    axes[1, 1].set_xticks(range(len(scaling_df)))
    axes[1, 1].set_xticklabels(scaling_df['model_size'], rotation=45, ha='right')
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('results/sa/qwen3-scaling-analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Scaling analysis plots saved to: results/sa/qwen3-scaling-analysis.png")

## Evaluate Selected Models on FLUKE Modifications

Test the best and worst performing models on key FLUKE modifications to understand robustness scaling.

In [ ]:
def evaluate_modified_set(ds, program, max_samples=30):
    """Evaluate on modified dataset with sample limit."""
    # Limit samples for scaling experiment
    limited_ds = ds[:max_samples] if len(ds) > max_samples else ds
    
    examples = [
        dspy.Example({ 
                      "text": remove_space(r['modified_text']), 
                      "original_text": remove_space(r['original_text']),
                      "label": int(r['label']),
                      "modified_label": int(r['label'])
                    }).with_inputs("text") 
        for r in limited_ds
    ]
    
    evaluate = Evaluate(
        devset=examples, 
        metric=eval_metric, 
        num_threads=1, 
        display_progress=True, 
        display_table=1, 
        return_outputs=True, 
        return_all_scores=True
    )
    
    return evaluate(program)

In [ ]:
# Select best and worst models for modification testing
if scaling_analysis and len(scaling_analysis) > 1:
    scaling_df_sorted = scaling_df.sort_values('accuracy')
    
    # Select models to test on modifications
    test_models = [
        scaling_df_sorted.iloc[0]['model_name'],   # Worst performing
        scaling_df_sorted.iloc[-1]['model_name'],  # Best performing
    ]
    
    # Add middle model if we have enough
    if len(scaling_df_sorted) >= 3:
        middle_idx = len(scaling_df_sorted) // 2
        test_models.insert(1, scaling_df_sorted.iloc[middle_idx]['model_name'])
    
    print(f"Testing modifications on selected models: {test_models}")
    
    # Key modifications to test
    test_modifications = ['typo_bias_100.json', 'capitalization_100.json', 'punctuation_100.json']
    json_files = glob.glob('../data/modified_data/sa/*_100.json')
    json_files = [f for f in json_files if any(mod in f for mod in test_modifications)]
    
    print(f"Testing modifications: {[f.split('/')[-1] for f in json_files]}")
    
    modification_results = []
    
    for model_name in test_models:
        if model_name not in model_results:
            continue
            
        print(f"\nTesting modifications for {model_name}...")
        model_config = model_results[model_name]['config']
        
        # Configure model
        lm = dspy.LM(model_config['id'], temperature=0, max_tokens=100)
        dspy.configure(lm=lm)
        qwen_sentiment = QwenSentimentModule()
        
        # Load original predictions for comparison
        original_pred_ds = model_results[model_name]['results_df']
        original_pred_ds['text'] = original_pred_ds['text'].apply(remove_space)
        
        for json_file in json_files:
            print(f"  Processing: {json_file.split('/')[-1]}")
            
            with open(json_file, 'r') as f:
                data = json.load(f)
            
            try:
                results = evaluate_modified_set(data, qwen_sentiment, max_samples=20)
                
                # Convert results to dataframe
                items = []
                for sample in results[1]:
                    item = {}
                    sentence = sample[0]['text']
                    label = sample[0]['label']
                    pred = sample[1]['label']
                    item['text'] = sentence
                    item['modified_label'] = label
                    pred_clean = extract_prediction(pred)
                    item['modified_pred'] = pred_clean
                    
                    original_text = sample[0]['original_text']
                    try:
                        original_text = original_text.encode('utf-8').decode('unicode-escape')
                    except:
                        pass
                        
                    item['original_label'] = sample[0]['label']
                    item['original_text'] = original_text
                    
                    # Find original prediction
                    matching_rows = original_pred_ds[original_pred_ds['text'] == original_text]
                    if not matching_rows.empty:
                        item['original_pred'] = matching_rows.iloc[0]['pred']
                    else:
                        item['original_pred'] = None
                        
                    item['raw_output'] = pred
                    items.append(item)
                
                df_result = pd.DataFrame(data=items)
                
                # Save results
                mod_name = json_file.split('/')[-1].replace('.json', '')
                output_filename = f"results/sa/{model_name}-0shot-{mod_name}.csv"
                df_result.to_csv(output_filename, index=False)
                
                # Store for analysis
                modification_results.append({
                    'model_name': model_name,
                    'model_size': model_config['size'],
                    'modification': mod_name,
                    'accuracy': results[0],
                    'samples': len(items)
                })
                
                print(f"    Accuracy: {results[0]:.3f}")
                
                time.sleep(3)
                
            except Exception as e:
                print(f"    Error: {e}")
                continue
else:
    print("Skipping modification testing - insufficient models evaluated")

## Final Analysis and Insights

In [ ]:
# Comprehensive scaling analysis
print("\n" + "="*80)
print("QWEN3 MODEL SCALING ANALYSIS - SENTIMENT CLASSIFICATION")
print("="*80)

if scaling_analysis:
    scaling_df = pd.DataFrame(scaling_analysis).sort_values('params')
    
    print("\nModel Performance Summary:")
    print("-" * 50)
    for idx, row in scaling_df.iterrows():
        print(f"{row['model_size']:12} | Acc: {row['accuracy']:.3f} | Time: {row['evaluation_time']:.1f}s | Throughput: {row['throughput']:.2f} samples/s")
    
    # Performance insights
    best_accuracy = scaling_df.loc[scaling_df['accuracy'].idxmax()]
    fastest_model = scaling_df.loc[scaling_df['throughput'].idxmax()]
    
    print(f"\nKey Insights:")
    print(f"• Best accuracy: {best_accuracy['model_size']} ({best_accuracy['accuracy']:.3f})")
    print(f"• Fastest model: {fastest_model['model_size']} ({fastest_model['throughput']:.2f} samples/s)")
    
    # Scaling trends
    if len(scaling_df) > 2:
        accuracy_range = scaling_df['accuracy'].max() - scaling_df['accuracy'].min()
        param_range_log = np.log10(scaling_df['params'].max()) - np.log10(scaling_df['params'].min())
        
        print(f"• Accuracy range: {accuracy_range:.3f} across {param_range_log:.1f} orders of magnitude")
        
        # Simple correlation analysis
        param_acc_corr = np.corrcoef(np.log10(scaling_df['params']), scaling_df['accuracy'])[0, 1]
        print(f"• Parameter-accuracy correlation: {param_acc_corr:.3f}")

# Modification robustness analysis
if 'modification_results' in locals() and modification_results:
    mod_df = pd.DataFrame(modification_results)
    print(f"\nRobustness Analysis (across {len(test_models)} selected models):")
    print("-" * 50)
    
    for modification in mod_df['modification'].unique():
        mod_subset = mod_df[mod_df['modification'] == modification]
        avg_accuracy = mod_subset['accuracy'].mean()
        std_accuracy = mod_subset['accuracy'].std()
        print(f"{modification:20} | Avg: {avg_accuracy:.3f} ± {std_accuracy:.3f}")
    
    print(f"\nModel size impact on robustness:")
    for model in test_models:
        if model in model_results:
            model_subset = mod_df[mod_df['model_name'] == model]
            if not model_subset.empty:
                avg_rob = model_subset['accuracy'].mean()
                original_acc = model_results[model]['accuracy']
                robustness_drop = original_acc - avg_rob
                print(f"  {model_results[model]['config']['size']:10} | Drop: {robustness_drop:+.3f} (orig: {original_acc:.3f} → mod: {avg_rob:.3f})")

print(f"\nExperiment Summary:")
print(f"• Total models tested: {len(model_results)}")
print(f"• Sample size per model: {SAMPLE_SIZE}")
print(f"• Modifications tested: {len(json_files) if 'json_files' in locals() else 0}")
print(f"• Results saved in: results/sa/")

print("\n" + "="*80)
print("Qwen3 Scaling Experiment Complete!")
print("="*80)

In [ ]:
print(f"\nFiles generated:")
print(f"1. Individual model results: results/sa/qwen3-*-0shot-sst2.csv")
print(f"2. Scaling analysis: results/sa/qwen3-scaling-analysis.csv")
print(f"3. Visualization: results/sa/qwen3-scaling-analysis.png")
print(f"4. Modification results: results/sa/qwen3-*-0shot-*_100.csv")

print(f"\nKey findings for Qwen3 scaling on FLUKE:")
print(f"• Model size impact on sentiment classification accuracy")
print(f"• Performance vs efficiency trade-offs across model sizes")
print(f"• Robustness to linguistic modifications at different scales")
print(f"• Optimal model size selection for FLUKE evaluation tasks")